# 02 · Analysis — what the registry can and cannot tell us

Every section states the **question**, the **method**, the **result** and its **limit**.

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from dnp_utils import PROCESSED, OUTPUTS

plt.rcParams.update({"figure.dpi":110,"axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.alpha":.25})
OUTPUTS.mkdir(exist_ok=True)
d = pd.read_parquet(PROCESSED/"projects_clean.parquet")
v = d[d.has_value].copy()                    # valued projects only
bil = lambda x,_: f"{x/1e12:.0f}"           # pesos -> trillions (billones COP)
print(len(d), "projects;", len(v), "with a value")

## 1. Portfolio composition by funding source

In [ ]:
g = d.groupby("TypeClean").agg(projects=("Bpin","size"), valued=("has_value","sum"))
g["total_value_trillion_COP"] = v.groupby("TypeClean").ValorTotalProyecto.sum()/1e12
g["median_value_million_COP"] = v.groupby("TypeClean").ValorTotalProyecto.median()/1e6
g = g.sort_values("projects", ascending=False).round(1)
g

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,3.8))
s = g.projects.head(4); s.sort_values().plot.barh(ax=ax[0], color="#4C78A8"); ax[0].set_title("Number of projects"); ax[0].set_ylabel("")
s = g.total_value_trillion_COP.head(4); s.sort_values().plot.barh(ax=ax[1], color="#F58518"); ax[1].set_title("Total value (trillion COP, valued projects only)"); ax[1].set_ylabel("")
plt.tight_layout(); plt.savefig(OUTPUTS/"01_composition_by_source.png", bbox_inches="tight"); plt.show()

**Reading:** Territorial projects dominate by count, but the picture by money can differ. **Limit:** value totals cover only projects with a value, and sums are driven by a few very large projects (next section).

## 2. Value is extremely skewed → use medians, not means

In [ ]:
x = v.ValorTotalProyecto
print(f"mean   : {x.mean()/1e6:,.0f} M COP")
print(f"median : {x.median()/1e6:,.0f} M COP")
top1 = x.sort_values(ascending=False).head(int(len(x)*0.01)).sum()/x.sum()
print(f"Top 1% of valued projects hold {top1:.0%} of total value")
fig, ax = plt.subplots(figsize=(7,3.6))
ax.hist(np.log10(x), bins=60, color="#4C78A8"); ax.set_xlabel("log10(project value in COP)"); ax.set_ylabel("projects")
ax.set_title("Project values span many orders of magnitude")
plt.savefig(OUTPUTS/"02_value_distribution.png", bbox_inches="tight"); plt.show()

## 3. Which sectors concentrate the money?

In [ ]:
sec = v.groupby("Sector").agg(projects=("Bpin","size"), total=("ValorTotalProyecto","sum"), median=("ValorTotalProyecto","median"))
sec["share_of_total"] = sec.total/sec.total.sum()
sec = sec.sort_values("total", ascending=False)
top = sec.head(10)
fig, ax = plt.subplots(figsize=(8,4.5))
ax.barh(top.index[::-1], top.total[::-1]/1e12, color="#F58518")
ax.set_xlabel("Total value (trillion COP)"); ax.set_title("Top 10 sectors by total value (valued projects, nominal pesos)")
plt.savefig(OUTPUTS/"03_top_sectors.png", bbox_inches="tight"); plt.show()
(top.assign(total_tn=(top.total/1e12).round(1), median_M=(top["median"]/1e6).round(0), share_pct=(top.share_of_total*100).round(1))[["projects","total_tn","median_M","share_pct"]])

**Limit:** this is *where the registry says the money is*, summed over projects of different years in nominal pesos. It is not an inflation-adjusted budget and not a measure of impact.

## 4. Pipeline: where do projects sit?

In [ ]:
pipe = pd.crosstab(d.TypeClean, d.StateGroup, normalize="index").round(3)
pipe = pipe.loc[["Territorial","SGR","PGN"]]
pipe

In [ ]:
ax = pipe[["Pre-execution","In execution","Finished","Rejected"]].plot.barh(stacked=True, figsize=(8,3.2), color=["#9ecae9","#4C78A8","#54A24B","#E45756"])
ax.set_xlabel("share of projects"); ax.set_ylabel(""); ax.set_title("Status mix by source — 'Finished' only exists for SGR")
ax.legend(loc="upper center", bbox_to_anchor=(.5,-.25), ncol=4, frameon=False)
plt.savefig(OUTPUTS/"04_pipeline_by_source.png", bbox_inches="tight"); plt.show()

**This chart is the main correction to version 1.** Finished projects only exist in SGR, so "x% of all projects are finished" is not a performance indicator. What can be studied is the SGR subset.

## 5. Do shorter projects finish more often? (SGR only, with a proper denominator)

**Method:** take SGR projects whose planned start is ≤ 2021 (so they had time to finish), group by planned duration, and compute the share `Finished` among *all* projects in the group — not only among those that finished.

Version 1 looked only at finished projects and concluded that "short projects are the most successful". That is **survivorship bias**: it never asked how many short projects *didn't* finish.

In [ ]:
s = d[(d.TypeClean=="SGR") & (d.StartYear<=2021) & (~d.horizon_suspect) & d.StateGroup.isin(["Pre-execution","In execution","Finished"])].copy()
s["duration_band"] = pd.cut(s.HorizonYears, [0,3,10,100], labels=["1-3 yrs","4-10 yrs",">10 yrs"])
r = s.groupby("duration_band", observed=True).agg(projects=("Bpin","size"), finished_share=("StateGroup", lambda x: (x=="Finished").mean()))
r.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.4))
ax.bar(r.index.astype(str), r.finished_share*100, color="#54A24B")
for i,(n,p) in enumerate(zip(r.projects, r.finished_share)): ax.text(i, p*100+1, f"{p:.0%}\n(n={n:,})", ha="center", fontsize=9)
ax.set_ylabel("% finished"); ax.set_ylim(0,100); ax.set_title("SGR projects started ≤2021: share finished by planned duration")
plt.savefig(OUTPUTS/"05_sgr_finished_by_duration.png", bbox_inches="tight"); plt.show()

**Reading and limits.** With the denominator included, the result is the *opposite* of version 1: the shortest projects (1–3 years) have the **lowest** share finished (about 51%), against about 87% for 4–10 years and 91% above 10. Version 1's "short projects are the most successful" came from counting only survivors. Still **not causal**: duration is a planned field, longer projects are probably larger/more complex, and the registry's "Finished" status depends on administrative closure that is reported with delay. I report the association and the cohort control (start ≤ 2021), nothing more.

## 6. Execution rate in SGR: payments vs current budget

Only SGR projects **currently in execution** with a positive current budget (`Vigente`). Finished projects are excluded on purpose: nearly all of them have paid ≈100%, which would make the metric trivial.

In [ ]:
e = d[(d.TypeClean=="SGR") & (d.StateGroup=="In execution") & (d.ValorVigenteProyecto>0)].copy()
e["pay_rate"] = e.ValorPagoProyecto/e.ValorVigenteProyecto
print("Projects:", len(e), "| median paid/budget:", round(e.pay_rate.median(),2),
      "| share with paid > 105% of budget:", round((e.pay_rate>1.05).mean(),3))
e["pay_rate"] = e.pay_rate.clip(upper=1.5)
es = e.groupby("Sector").agg(projects=("Bpin","size"), median_rate=("pay_rate","median"))
es = es[es.projects>=50].sort_values("median_rate")
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(es.index, es.median_rate*100, color="#4C78A8"); ax.set_xlabel("median % of current budget already paid")
ax.set_title("SGR projects in execution: payment rate by sector (≥50 projects)")
plt.savefig(OUTPUTS/"06_sgr_execution_by_sector.png", bbox_inches="tight"); plt.show()
es.round(2)

**Limits.** Payments lag by nature and projects are at different stages, so low rates partly reflect timing. About 12% of projects show payments above their current budget, which suggests `Vigente` is not always updated; ratios are capped at 150%. The sector with a median above 100% (*Ciencia, Tecnología e Innovación*) is more likely a budget-update issue than genuine over-spending. Sector differences here are descriptive, not a ranking of who executes well.

## 7. Cost per beneficiary — why I do NOT publish a ranking

In [ ]:
c = d[d.benef_valid.fillna(False) & d.has_value].copy()
c["cost_per_benef"] = c.ValorTotalProyecto/c.TotalBeneficiario.astype(float)
print("Projects usable:", len(c), "of", len(d))
q = c.cost_per_benef.quantile([.01,.25,.5,.75,.99])
print((q/1e3).round(1).to_string(), "\n(thousand COP per beneficiary)")

Even after removing impossible values, cost per beneficiary spans several orders of magnitude (the 1st percentile is about 100 COP per beneficiary and the 99th about 45 million, a ratio of roughly 450,000×), because "beneficiaries" is self-reported and defined differently by project type (a school vs. a highway vs. a regulatory body). **Conclusion: this column cannot support an "efficiency" ranking.** Version 1 did exactly that and it should be read as a documented mistake, not a result.

## Summary of what the data supports

1. Territorial projects are the large majority by count; value is extremely concentrated (the top 1% of valued projects hold about 75% of total value).
2. Status is not comparable across sources; completion only makes sense within SGR.
3. Within SGR, longer planned duration goes with a *higher* share finished; this reverses v1's conclusion once denominators are used (association, not causation).
4. Beneficiary-based efficiency is **not** supportable with this dataset.